# 🎓 Predicting Student Mental Health Score from Social Media Usage

**Problem:** We're predicting a student's `Mental_Health_Score` (a continuous score from ~3 to ~10) using their social media habits, study time, sleep, physical activity, and stress level. This is a **regression** problem since the target is a number, not a category.

**Dataset:** `Student_Social_Media_And_Mental_Health_Impact.csv` — 5,000 students, 13 columns covering demographics (Age, Gender, Country), platform usage (Avg_Daily_Usage_Hours, Daily_Unlocks, Most_Used_Platform), lifestyle (Study_Hours, Sleep_Hours_Per_Night, Physical_Activity_Hours), and Stress_Level.



## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load the Dataset

We load the CSV and take a first look at its shape and the first few rows

In [ ]:
df = pd.read_csv('Student Social Media And Mental Health Impact.csv')

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

📌 Zero null rows and 2 duplicates

In [ ]:
df[df.duplicated(keep = False)]

In [ ]:
df.describe()

📌 Most columns look reasonable — but look closely at `Physical_Activity_Hours`: the **minimum value is -0.4**. Negative hours aren't physically possible, so this is a data-entry glitch, not a real value

## 4. Exploratory Data Analysis (EDA)

### 4.1 — Distribution of the target (`Mental_Health_Score`)


In [ ]:
sns.histplot(df['Mental_Health_Score'],kde=True)

### 4.2 — Correlation heatmap

Which numeric features actually move together with the target?

In [ ]:
sns.heatmap(df.corr(numeric_only = True),annot = True)

### 4.3 — Stress Level vs Mental Health Score


In [ ]:
df["Stress_Level"].unique()

In [ ]:
order = ['Low', 'Medium', 'High', 'Very High']
sns.boxplot(x = 'Stress_Level', y = 'Mental_Health_Score', data = df, order = order)

### 4.4 — Daily Usage Hours vs Mental Health Score


In [ ]:
sns.scatterplot(x='Avg_Daily_Usage_Hours', y = 'Mental_Health_Score', data = df)

### 4.5 — Sleep Hours vs Mental Health Score

In [ ]:
sns.scatterplot(x = 'Sleep_Hours_Per_Night', y = 'Mental_Health_Score', data = df)

### 4.6 — Most Used Platform (count)

In [ ]:
df['Most_Used_Platform'].value_counts()

In [ ]:
sns.countplot(x = 'Most_Used_Platform',data = df, order = df['Most_Used_Platform'].value_counts().index)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

**Checking Outliers**

In [ ]:
num_features = df.select_dtypes(include = 'number')
Q1 = num_features.quantile(0.25)
Q3 = num_features.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = (num_features <  lower_bound) | (num_features > upper_bound)
print(outliers.sum())


## 5. Data Cleaning


In [ ]:
df.describe()

In [ ]:
df.drop_duplicates(inplace = True)
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower = 0)

## 6. Skewness


In [ ]:
num_cols = df.select_dtypes(include = 'number')
num_cols.skew()

## 7. Feature Engineering

In [ ]:
top_countries = df['Country'].value_counts().index[:10].to_list()

In [ ]:
def group_countries(country):
  if country in top_countries:
    return country
  else:
    return 'Other'

In [ ]:
df['Grouped_country'] = df['Country'].apply(group_countries)

In [ ]:
df['Grouped_country'].value_counts()

📌 **Result:** we went from 111 raw categories down to 11 manageable ones (10 countries + "Other")

## 8. Train-Test Split


In [ ]:
from sklearn.model_selection import train_test_split
skewed_col = ['Study_Hours']
other_numerical_cols = ['Age','Avg_Daily_Usage_Hours','Daily_Unlocks','Physical_Activity_Hours','Sleep_Hours_Per_Night']
ordinal_col = ["Stress_Level"]
normal_col = ['Gender','Academic_Level', 'Most_Used_Platform','Purpose_Of_Use', 'Grouped_country']
feature_col = skewed_col + other_numerical_cols + ordinal_col + normal_col
x = df[feature_col]
y = df['Mental_Health_Score']
x_train, x_test, y_train, y_test = train_test_split(x, y,test_size=0.30,random_state=42)

## 10. Preprocessing using ColumnTransformer

Our columns need **different treatment**:

- `Study_Hours` (the skewed one) → impute → `log1p` transform → scale
- The other numeric columns → impute → scale (no skew to fix)
- `Stress_Level` → impute → `OrdinalEncoder` with an **explicit order**
- The nominal columns → impute → `OneHotEncoder`


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

skew_pipeline = Pipeline(steps = [
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())
])

plain_numeric_pipeline = Pipeline(steps=[
    ('scale', StandardScaler())
])

ordinal_pipeline = Pipeline(steps = [
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High']]))
])

nominal_pipeline = Pipeline(steps = [
    ('encode', OneHotEncoder(handle_unknown = 'ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('skew_pipeline', skew_pipeline, skewed_col),
    ('plain_numeric_pipeline', plain_numeric_pipeline, other_numerical_cols),
    ('ordinal_pipeline', ordinal_pipeline, ordinal_col),
    ('nominal_pipeline', nominal_pipeline, normal_col)
])

## 12. Model Building

### 12.1 — Baseline: Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
lr_pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
lr_pipeline.fit(x_train, y_train)
lr_preds = lr_pipeline.predict(x_test)
lr_preds_train = lr_pipeline.predict(x_train)
lr_r2_testing = r2_score(y_test,lr_preds)
lr_r2_training = r2_score(y_train,lr_preds_train)
lr_mae = mean_absolute_error(y_test,lr_preds)

print(f"Accuracy of Training {lr_r2_training}")
print(f"Accuracy of Testing {lr_r2_testing}")
print(f"MAE {lr_mae}")

### 12.2 — Random Forest (default settings)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps = [
    ('preprocessor',preprocessor),
    ('random forest', RandomForestRegressor(random_state = 42))
])

rf_pipeline.fit(x_train, y_train)
rf_preds = rf_pipeline.predict(x_test)
rf_preds_train = rf_pipeline.predict(x_train)

rf_r2_testing = r2_score(y_test, rf_preds)
rf_r2_training = r2_score(y_train, rf_preds_train)
rf_mae = mean_absolute_error(y_test, rf_preds)

print(f"Accuracy of Training {rf_r2_training}")
print(f"Accuracy of Testing {rf_r2_testing}")
print(f"MAE {rf_mae}")


### 12.3 — Hyperparameter Tuning the Random Forest

When tuning a `Pipeline`, each parameter needs to be prefixed with the step name (`regressor__`) so `RandomizedSearchCV` knows which part of the pipeline it belongs to. We use `RandomizedSearchCV` instead of a full grid search — it samples a fixed number of combinations instead of testing every single one, which is far faster while still finding a strong set of parameters.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'random forest__n_estimators': [100, 200, 300],
    'random forest__max_depth': [5, 10,15],
    'random forest__min_samples_split': [2, 5, 10],
    'random forest__min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator = rf_pipeline,
    param_distributions = param_grid,
    n_iter = 15,
    cv = 5,
    scoring = 'r2',
    random_state = 42,
    n_jobs = -1
)

random_search.fit(x_train,y_train)


In [ ]:
random_search.best_params_

In [ ]:
rf_best_pipeline = random_search.best_estimator_
rf_best_preds = rf_best_pipeline.predict(x_test)

In [ ]:
print(f"r2 score for random forest after hyper parameter tuning {r2_score(y_test,rf_best_preds)}")
print(f"Mae score for random forest after hyper parameter tuning {mean_absolute_error(y_test,rf_best_preds)}")

## 13. Model Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error

# Calculate RMSE for Linear Regression
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

# Calculate RMSE for default Random Forest
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

# Calculate RMSE for tuned Random Forest
rf_tuned_preds = random_search.best_estimator_.predict(x_test)
rf_tuned_rmse = np.sqrt(mean_squared_error(y_test, rf_tuned_preds))
rf_tuned_training_preds = random_search.best_estimator_.predict(x_train)
r2_tuned_training = r2_score(y_train, rf_tuned_training_preds)
rf_tuned_mae = mean_absolute_error(y_test, rf_tuned_preds)
rf_tuned_r2 = r2_score(y_test, rf_tuned_preds)


# Create a DataFrame to consolidate results
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (default)', 'Random Forest (tuned)'],
    'R2': [lr_r2_testing, rf_r2_testing, rf_tuned_r2],
    'Training R2': [lr_r2_training, rf_r2_training, r2_tuned_training],
    'MAE': [lr_mae, rf_mae, rf_tuned_mae],
    'RMSE': [lr_rmse, rf_rmse, rf_tuned_rmse]
})

print(results)


In [ ]:
results

## 14. Save the Model

We save the **entire pipeline** — preprocessing and model together — as a single file, not just the raw model. **Why:** if we only saved the Random Forest object, our FastAPI backend would need to manually recreate every encoding and scaling step by hand at prediction time, which is fragile and easy to get wrong. Saving the full pipeline means the backend just loads one file and calls `.predict()` on raw input — the pipeline handles everything internally, exactly the same way it did during training.

In [ ]:
import joblib
joblib.dump(rf_pipeline, 'Mental_Health_Model.pkl')


#**Next steps (Part 2):**
build a FastAPI backend with Pydantic request validation that loads `mental_health_pipeline.pkl` and exposes a `/predict` endpoint, then connect a simple frontend, then deploy both so this becomes a live, clickable demo.
